# Persistence in LangGraph

## 1. Why Do We Need Persistence? (The Reason)

* **Default Stateless Behavior:** In LangGraph, when an execution run reaches `END`, the graph run terminates and the state in memory is discarded. A second call to `graph.invoke()` starts with a completely fresh, empty state.
* **The Multi-Turn Problem:** Without persistence, an LLM chatbot cannot remember previous interactions across distinct `invoke()` calls, even if `add_messages` reducer is used (because `add_messages` only appends messages within a single graph run).
* **How Checkpointing Solves It:**
  1. A **Checkpointer** saves snapshots of the graph state at each step into a storage backend.
  2. Each conversation or session is assigned a unique `thread_id` via `config = {"configurable": {"thread_id": "..."}}`.
  3. When a new turn is sent with the same `thread_id`, LangGraph reloads the prior state, appends new messages, and resumes seamlessly.
  4. Enables critical agent capabilities: **multi-turn chat memory**, **human-in-the-loop (pausing/resuming)**, and **fault tolerance/time-travel**.

## 2. Different Types of Persistence in LangGraph (Short Overview)

| Checkpointer | Package | Storage Location | Best Used For |
| :--- | :--- | :--- | :--- |
| **`MemorySaver`** | `langgraph.checkpoint.memory` | In-Memory (RAM) | Quick prototyping, testing, and notebooks. *(State is lost if the process restarts)* |
| **`SqliteSaver`** | `langgraph-checkpoint-sqlite` | Local file (`.db`) | Lightweight single-server apps, local desktop bots without extra DB infra. |
| **`MongoDBSaver`** | `langgraph-checkpoint-mongodb` | MongoDB cluster | Document/NoSQL architectures, scalable cloud services, native JSON persistence. |
| **`PostgresSaver`** | `langgraph-checkpoint-postgres` | PostgreSQL database | Enterprise production, ACID compliance, high concurrency, multi-instance setups. |
| **`RedisSaver`** | `langgraph-checkpoint-redis` | Redis cache/store | High-throughput session handling, low latency, auto-expiry via TTL. |

## 3. Standard Code Implementation Pattern

```python
# 1. Import the desired checkpointer
from langgraph.checkpoint.memory import MemorySaver  # or MongoDBSaver, PostgresSaver, etc.

# 2. Instantiate checkpointer
checkpointer = MemorySaver()

# 3. Compile graph with checkpointer
chatbot = graph.compile(checkpointer=checkpointer)

# 4. Invoke using configurable thread_id
config = {"configurable": {"thread_id": "user_session_1"}}
response = chatbot.invoke({"messages": [HumanMessage(content="Hi")]}, config=config)
```